In [ ]:
import pandas as pd
import glob
import os

# Configuration
data_folder_1 = "Consurf/"
data_folder_2 = "Phosphosites/"

# Collect all relevant csv files
consurf_files = glob.glob(os.path.join(data_folder_1, "*msa_aa_variety_percentage.csv"))
phospho_files = glob.glob(os.path.join(data_folder_2, "*Phosphosites.csv"))

# Helper function: find matching phosphosite file for a given protein
def find_matching_phospho(consurf_file, phospho_files):
    base_name = os.path.basename(consurf_file).split("_msa_aa_variety_percentage")[0]
    for p in phospho_files:
        if base_name in os.path.basename(p):
            return p
    return None

# Process each ConSurf–Phosphosite pair
for consurf_file in consurf_files:
    phospho_file = find_matching_phospho(consurf_file, phospho_files)
    
    if not phospho_file:
        print(f"No matching phosphosite file found for {consurf_file}")
        continue
    
    print(f"Processing pair: {os.path.basename(consurf_file)} + {os.path.basename(phospho_file)}")

    # Load ConSurf (skip description)
    consurf_df = pd.read_csv(consurf_file, skiprows=4)
    consurf_df.columns = [c.strip() for c in consurf_df.columns]

    # Load Phosphosite
    phospho_df = pd.read_csv(phospho_file)
    phospho_positions = phospho_df.iloc[:, 0].astype(int).tolist()

    # Filter by positions
    filtered = consurf_df[consurf_df["pos"].isin(phospho_positions)]

    # Count per ConSurf grade
    counts = filtered["ConSurf grade"].value_counts().sort_index()

    # Save to output CSV
    base_name = os.path.basename(consurf_file).split("_msa_aa_variety_percentage")[0]
    out_file = f"Figure_S8B_Phosphosite_Count_by_ConSurf_Grade_{base_name}.csv"
    counts.to_csv(out_file, header=["Phosphosite_Count"])
    
    print(f"Saved: {out_file}")

print("All protein pairs processed successfully!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import glob

# Define gradient generator
def make_gradient(hex_color, n=10):
    """Return a list of n gradient colors from white to given hex color."""
    cmap = mcolors.LinearSegmentedColormap.from_list("grad", ["#ffffff", hex_color])
    return [mcolors.rgb2hex(cmap(i / (n - 1))) for i in range(n)]

# Define base colors per protein family
protein_colors = {
    "TSC22D1": make_gradient("#ffe5b6"),
    "TSC22D2": make_gradient("#ffe5b6"),
    "TSC22D3": make_gradient("#ffe5b6"),
    "TSC22D4": make_gradient("#ffe5b6"),
    "WNK1": make_gradient("#49c1bb"),
    "WNK2": make_gradient("#49c1bb"),
    "WNK3": make_gradient("#49c1bb"),
    "WNK4": make_gradient("#49c1bb"),
    "NRBP1": make_gradient("#bababa"),
    "NRBP2": make_gradient("#bababa"),
    "STK39": make_gradient("#6dabc6"),
    "OXSR1": make_gradient("#6dabc6"),
}

# Load all result CSVs
files = glob.glob("Figure_S8B_Phosphosite_Count_by_ConSurf_Grade_*.csv")

dfs = []
for f in files:
    protein = f.split("_Grade_")[1].replace(".csv", "")
    df = pd.read_csv(f)
    df.columns = ["ConSurf_grade", protein]
    dfs.append(df.set_index("ConSurf_grade"))

# Merge all into combined table
combined = pd.concat(dfs, axis=1).fillna(0)

# Normalize each column (sum = 1)
combined_norm = combined.div(combined.sum(axis=0), axis=1)

# Sort columns in desired family order
desired_order = [
    "TSC22D1", "TSC22D2", "TSC22D3", "TSC22D4",
    "WNK1", "WNK2", "WNK3", "WNK4",
    "NRBP1", "NRBP2",
    "STK39", "OXSR1"
]
combined_norm = combined_norm[[p for p in desired_order if p in combined_norm.columns]]

# Plot setup
fig, ax = plt.subplots(figsize=(3, 3))
bar_width = 1.0
x = np.arange(len(combined_norm.columns))

# Draw stacked bars
for i, protein in enumerate(combined_norm.columns):
    y_bottom = 0
    values = combined_norm[protein]
    grades = values.index.astype(int)

    grad = protein_colors.get(protein, make_gradient("#cccccc"))
    for grade in grades:
        val = values.loc[grade]
        color = grad[grade] if grade < len(grad) else grad[-1]
        ax.bar(
            i, val,
            bar_width,
            bottom=y_bottom,
            color=color,
            edgecolor="black",   # black boundary between grades
            linewidth=0.25,
            align="edge"
        )
        y_bottom += val

# Remove outer spines for clean heatmap-like look
for spine in ax.spines.values():
    spine.set_visible(False)

# Aesthetic tweaks
ax.set_xticks(x)  # <-- FIXED: center labels under bars
ax.set_xticklabels(combined_norm.columns, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("Fraction of Phosphosites", fontsize=12)
ax.set_xlabel("")
ax.set_title("Normalized Phosphosite Distribution by ConSurf Grade", fontsize=10)
ax.set_ylim(0, 1)

# Remove outer margins
plt.margins(0)
plt.tight_layout(pad=0.1)
plt.savefig("Figure_S8B_Normalized_Phosphosite_Distribution_by_Consurf.pdf", format="pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from matplotlib.patches import Rectangle

def make_gradient(hex_color, n=10):
    """Return list of n colors from white to given color."""
    cmap = mcolors.LinearSegmentedColormap.from_list("", ["white", hex_color])
    return [mcolors.rgb2hex(cmap(i / (n - 1))) for i in range(n)]

# Define colors per family
family_colors = {
    "TSC22D": "#ffe5b6",
    "WNK": "#49c1bb",
    "NRBP": "#bababa",
    "STK39/OXSR1": "#6dabc6"
}

grades = list(range(10))  # 0–9

# Create figure
fig, axes = plt.subplots(len(family_colors), 1, figsize=(4, 2), constrained_layout=True)

for ax, (family, hex_color) in zip(axes, family_colors.items()):
    colors = make_gradient(hex_color, n=10)

    # Draw discrete color blocks
    for i, color in enumerate(colors):
        ax.bar(
            i, 1,
            color=color,
            width=1.2,
            edgecolor="black",
            linewidth=0.8,
            align="edge"
        )

    # Add outer rectangle border for the whole row
    # Rectangle coordinates: (x, y, width, height)
    outer_rect = Rectangle(
        (0, 0),             # bottom-left corner
        10, 1,              # width and height (10 grades × height 1)
        linewidth=1.0,
        edgecolor="black",
        facecolor="none"
    )
    ax.add_patch(outer_rect)

    # Ticks & labels
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 1)
    ax.set_xticks(np.arange(0.5, 10.5, 1))
    ax.set_xticklabels(grades, fontsize=8)
    ax.set_yticks([])
    ax.set_ylabel(family, rotation=0, labelpad=65, fontsize=8, va="center")

    # Clean frame
    for spine in ax.spines.values():
        spine.set_visible(False)

# Plot & Save
plt.savefig("Figure_S8B_Color_legend.pdf", format="pdf", dpi=300, bbox_inches="tight", transparent=True)
plt.show()